# Sheridan — SDXL InstantID setup

This notebook prepares the existing `/notebooks/ComfyUI` installation for HerAndHim's bundled `sdxl-instantid` workflow. It does **not** train a LoRA or modify your SheridanV2 source code.

Run the cells in order. The first cell is inspection only. The model-download cell needs roughly **9 GB of free disk space** and downloads about 9 GB. If space is tight, stop before that cell and share the preflight output.

The selected SDXL checkpoint is Segmind SSD-1B (Apache 2.0), a smaller SDXL-derived checkpoint intended to be more practical on an 8 GB GPU than the full SDXL base. It is still not guaranteed to fit every ComfyUI configuration; we'll validate with a low-VRAM test before connecting the phone app.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, platform

COMFY = Path('/notebooks/ComfyUI')
print('Python:', sys.executable)
print('Python version:', platform.python_version())
print('ComfyUI folder exists:', COMFY.exists())
print('ComfyUI main.py exists:', (COMFY / 'main.py').exists())
print('Disk free (GB):', round(shutil.disk_usage('/notebooks').free / (1024**3), 2))

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
except Exception as e:
    print('PyTorch check:', repr(e))

if not COMFY.exists() or not (COMFY / 'main.py').exists():
    raise RuntimeError('Expected /notebooks/ComfyUI/main.py was not found. Stop here and send this output.')


## 2. Install the InstantID custom node and Python dependencies

This installs the native ComfyUI InstantID node pack and CPU InsightFace runtime (the workflow uses InsightFace on CPU). It leaves your existing model folders and other custom nodes untouched.


In [ ]:
CUSTOM = COMFY / 'custom_nodes'
NODE = CUSTOM / 'ComfyUI_InstantID'
CUSTOM.mkdir(parents=True, exist_ok=True)

if not NODE.exists():
    subprocess.run(['git', 'clone', 'https://github.com/cubiq/ComfyUI_InstantID.git', str(NODE)], check=True)
else:
    print('InstantID node folder already exists; leaving it in place:', NODE)

# Install into this notebook's Python. If ComfyUI is running under a different
# Python environment, stop and share the preflight output before restarting it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'cython<3', 'onnxruntime==1.20.1', 'insightface==0.7.3'], check=True)
print('Node and dependencies installed. Restart ComfyUI after all setup cells finish.')


## 3. Download the model files

This is the large-download step. It checks free disk space first and skips any file that is already present. Approximate total: 9 GB (SSD-1B 4.47 GB, InstantID adapter 1.69 GB, InstantID ControlNet 2.50 GB, antelopev2 0.36 GB).


In [ ]:
import zipfile
from huggingface_hub import hf_hub_download

free_gb = shutil.disk_usage('/notebooks').free / (1024**3)
if free_gb < 10:
    raise RuntimeError(f'Only {free_gb:.1f} GB free. Need at least 10 GB before downloading. Stop and share this result.')

folders = {
    'checkpoints': COMFY / 'models' / 'checkpoints',
    'instantid': COMFY / 'models' / 'instantid',
    'controlnet': COMFY / 'models' / 'controlnet' / 'instantid',
    'insightface': COMFY / 'models' / 'insightface' / 'models' / 'antelopev2',
    'cache': COMFY / 'models' / '_sheridan_download_cache',
}
for folder in folders.values(): folder.mkdir(parents=True, exist_ok=True)

def fetch(repo, filename, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1024*1024:
        print('Already present:', dest)
        return dest
    print('Downloading:', filename)
    downloaded = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=str(folders['cache'])))
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(downloaded), str(dest))
    print('Saved:', dest, 'GB:', round(dest.stat().st_size/(1024**3), 2))
    return dest

fetch('segmind/SSD-1B', 'SSD-1B.safetensors', folders['checkpoints'] / 'SSD-1B.safetensors')
fetch('InstantX/InstantID', 'ip-adapter.bin', folders['instantid'] / 'ip-adapter.bin')
fetch('InstantX/InstantID', 'ControlNetModel/diffusion_pytorch_model.safetensors', folders['controlnet'] / 'diffusion_pytorch_model.safetensors')

archive = Path(hf_hub_download(repo_id='MonsterMMORPG/tools', filename='antelopev2.zip', local_dir=str(folders['cache'])))
expected = {'1k3d68.onnx', '2d106det.onnx', 'genderage.onnx', 'glintr100.onnx', 'scrfd_10g_bnkps.onnx'}
present = {p.name for p in folders['insightface'].glob('*.onnx')}
if not expected.issubset(present):
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            name = Path(member).name
            if name in expected and name not in present:
                with z.open(member) as src, open(folders['insightface'] / name, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
print('antelopev2 files:', sorted(p.name for p in folders['insightface'].glob('*.onnx')))
print('Model download stage complete.')


## 4. Validate the installation

This checks that all required files exist and prints the checkpoint name to select in HerAndHim. It does not start ComfyUI or change your HerAndHim configuration.


In [ ]:
required = [
    folders['checkpoints'] / 'SSD-1B.safetensors',
    folders['instantid'] / 'ip-adapter.bin',
    folders['controlnet'] / 'diffusion_pytorch_model.safetensors',
    folders['insightface'] / '1k3d68.onnx',
    folders['insightface'] / '2d106det.onnx',
    folders['insightface'] / 'genderage.onnx',
    folders['insightface'] / 'glintr100.onnx',
    folders['insightface'] / 'scrfd_10g_bnkps.onnx',
]
for p in required: print(('OK   ' if p.exists() and p.stat().st_size > 0 else 'MISS '), p)
missing = [str(p) for p in required if not p.exists() or p.stat().st_size == 0]
if missing: raise RuntimeError('Missing required files; do not start the workflow yet.')
print('\nHerAndHim image settings to use after ComfyUI is running:')
print('skills.comfyui.checkpoint = SSD-1B.safetensors')
print('skills.comfyui.identityWorkflow = sdxl-instantid')
print('ComfyUI API = http://127.0.0.1:8188 (same host only; Docker networking must be checked separately)')
print('\nRestart ComfyUI after setup. Then we will verify /object_info and run a test image before changing the phone connection.')
